# Knowledge Vault Agent — Demo

This notebook demonstrates calling the Knowledge Vault Cortex Agent via REST API,
including org-based disambiguation of shared abbreviations (e.g., "CR").

Uses a connection from `~/.snowflake/config.toml` and the
thread API for multi-turn context management.

In [2]:
import json
import tomllib
from pathlib import Path

import requests

# Load connection config from ~/.snowflake/config.toml
config_path = Path.home() / ".snowflake" / "config.toml"
with open(config_path, "rb") as f:
    config = tomllib.load(f)

# Update this to match your connection name in config.toml
CONNECTION_NAME = "parker_demo"
conn = config["connections"][CONNECTION_NAME]
ACCOUNT = conn["account"]
PAT = conn["password"]  # PAT token stored as password

# Derive the account URL (underscores become hyphens in the hostname)
ACCOUNT_URL = f"https://{ACCOUNT.replace('_', '-').lower()}.snowflakecomputing.com"

# Agent location
AGENT_DB = "DOC_KNOWLEDGE_GRAPH_DEMO"
AGENT_SCHEMA = "SERVING"
AGENT_NAME = "DOC_KNOWLEDGE_GRAPH_AGENT"

AGENT_ENDPOINT = f"{ACCOUNT_URL}/api/v2/databases/{AGENT_DB}/schemas/{AGENT_SCHEMA}/agents/{AGENT_NAME}:run"
THREAD_ENDPOINT = f"{ACCOUNT_URL}/api/v2/cortex/threads"

HEADERS = {
    "Content-Type": "application/json",
    "Accept": "application/json",
    "Authorization": f"Bearer {PAT}",
}

print(f"Account: {ACCOUNT}")
print(f"Agent:   {AGENT_DB}.{AGENT_SCHEMA}.{AGENT_NAME}")
print(f"Endpoint: {AGENT_ENDPOINT}")

Account: SFSENORTHAMERICA-PERICKSON_AWS1
Agent:   DOC_KNOWLEDGE_GRAPH_DEMO.SERVING.DOC_KNOWLEDGE_GRAPH_AGENT
Endpoint: https://sfsenorthamerica-perickson-aws1.snowflakecomputing.com/api/v2/databases/DOC_KNOWLEDGE_GRAPH_DEMO/schemas/SERVING/agents/DOC_KNOWLEDGE_GRAPH_AGENT:run


In [3]:
def create_thread() -> str:
    """Create a new conversation thread. Returns the thread_id."""
    resp = requests.post(
        THREAD_ENDPOINT,
        headers=HEADERS,
        json={"origin_application": "dkg_demo"}
    )
    resp.raise_for_status()
    thread_data = resp.json()
    return thread_data["thread_id"]


def call_agent(message: str, thread_id: str, parent_message_id: str = "0", system_message: str = None) -> tuple[str, str]:
    """Send a message to the agent within a thread.
    
    Args:
        message: The user's question
        thread_id: The conversation thread ID
        parent_message_id: ID of the previous message in the thread (\"0\" for first message)
        system_message: Optional system prompt (e.g., user's team for disambiguation)
    
    Returns:
        Tuple of (response_text, message_id) — use message_id as parent_message_id for next turn
    """
    messages = []
    if system_message:
        messages.append({"role": "system", "content": [{"type": "text", "text": system_message}]})
    messages.append({"role": "user", "content": [{"type": "text", "text": message}]})
    
    payload = {
        "model": f"{AGENT_DB}.{AGENT_SCHEMA}.{AGENT_NAME}",
        "thread_id": thread_id,
        "parent_message_id": parent_message_id,
        "messages": messages,
    }
    
    resp = requests.post(AGENT_ENDPOINT, headers=HEADERS, json=payload)
    resp.raise_for_status()
    
    # Parse SSE response
    full_text = ""
    message_id = parent_message_id
    current_event = ""
    
    for line in resp.text.strip().split("\n"):
        if line.startswith("event:"):
            current_event = line[6:].strip()
        elif line.startswith("data:"):
            try:
                data = json.loads(line[5:])
                if current_event == "response.text.delta" and "text" in data:
                    full_text += data["text"]
                if current_event == "metadata":
                    meta = data.get("metadata", data)
                    if meta.get("role") == "assistant" and "message_id" in meta:
                        message_id = str(meta["message_id"])
            except json.JSONDecodeError:
                pass
    
    return full_text, message_id

print("Helper functions ready.")

Helper functions ready.


---
## Example Conversation Thread: Activation Rate Deep-Dive

This shows a real multi-turn conversation with the agent, starting from a basic
concept lookup and drilling into relationships and data sources.

In [4]:
# Create a new thread for this conversation
thread_id = create_thread()
print(f"Thread: {thread_id}")

Thread: 58553665177


In [5]:
# Turn 1: Basic concept lookup
print("USER: What is activation rate?")
print("=" * 60)
response, msg_id = call_agent(
    "What is activation rate?",
    thread_id,
    system_message="The user is on the Analytics team."
)
print(response)

USER: What is activation rate?
Activation Rate is a **Metric** that measures the percentage of trial users who complete onboarding and become paying subscribers.

**Calculation**

Activation Rate = Activated Subscribers / Qualified Trials Ã 100

**Key things to know**
- The denominator uses **qualified trials**, not total sign-ups. A trial only qualifies once the user completes at least one core workflow (e.g., connects a data source or creates a dashboard).
- Activation is confirmed when the **first subscription payment is processed**, not at the point of plan selection.
- Rates vary significantly by plan tier (Starter vs. Professional), so you should always report Activation Rate alongside that dimension.

**Common pitfalls**
- Don't confuse it with **Conversion Rate**, which measures a broader funnel (all sign-ups â paying subscribers). Activation Rate starts at qualified trials, whereas Conversion Rate includes all sign-ups â even those who never log in after registration.
- S

In [6]:
# Turn 2: Follow up — ask for the full concept note
print("USER: Can you pull the full concept note for Activation Rate?")
print("=" * 60)
response, msg_id = call_agent(
    "Can you pull the full concept note for Activation Rate?",
    thread_id,
    parent_message_id=msg_id
)
print(response)

USER: Can you pull the full concept note for Activation Rate?
Here is the full concept note for **Activation Rate** (`activation-rate`), reproduced in its entirety:

---

**Concept:** Activation Rate
**Kind:** Metric

Activation Rate measures the percentage of trial users who complete onboarding and become paying subscribers.

**Calculation**

```
Activation Rate = Activated Subscribers / Qualified Trials Ã 100
```

**Key Considerations**
- The denominator uses **qualified trials**, not total sign-ups. A trial qualifies once the user completes at least one core workflow (e.g., connects a data source or creates a dashboard).
- Activation is confirmed when the first subscription payment is processed, not at the point of plan selection.
- Rates vary significantly by plan tier (Starter vs. Professional) and should always be reported with that dimension.

**Common Pitfalls**
- Do not confuse with **Conversion Rate**, which measures a broader funnel (all sign-ups â paying subscribers).
- 

In [7]:
# Turn 3: Follow up — ask about relationships to data sources and business rules
print("USER: How is Activation Rate related to its underlying data sources and business rules?")
print("=" * 60)
response, msg_id = call_agent(
    "How is Activation Rate related to its underlying data sources and business rules?",
    thread_id,
    parent_message_id=msg_id
)
print(response)

USER: How is Activation Rate related to its underlying data sources and business rules?
Here's how Activation Rate connects to the data sources and business rules in the knowledge vault. Note that the vault doesn't define a dedicated "trial qualification" business rule or a product-events data source, so some of the mechanics the metric relies on aren't backed by their own notes â I've flagged those gaps below.

**The metric's own definition sets up the dependencies**

Activation Rate = Activated Subscribers / Qualified Trials Ã 100. Two things drive it: (1) a trial becomes a *qualified trial* once a user completes at least one core workflow (e.g., connects a data source or creates a dashboard), and (2) activation is confirmed when the *first subscription payment is processed* â not at plan selection.

**Data source: Salesforce**

Salesforce is the vault's documented source of truth for the sales funnel "from lead creation through subscription activation." The confirmed-subscripti

---
## Disambiguation Examples

These examples show how the same abbreviation resolves differently based on
the user's team context provided in the system prompt.

### Analytics Team Asks About "CR"

The system prompt tells the agent the user is on the Analytics team.
"CR" should resolve to **Completion Rate**.

In [8]:
analytics_thread = create_thread()
analytics_response, analytics_msg_id = call_agent(
    "What is CR and how do we measure it?",
    analytics_thread,
    system_message="The user is on the Analytics team."
)
print("ANALYTICS TEAM:")
print(analytics_response)

ANALYTICS TEAM:


I'm interpreting "CR" as **Completion Rate**, which is how the Analytics and Customer Success teams use the abbreviation. Note that "CR" has other meanings in this organization â the Finance team uses it for **Credit Rating** â and there's also a distinct **Conversion Rate** metric. Let me know if you meant one of those instead.

## Completion Rate (CR)

**What it is:** Completion Rate measures the percentage of customers in a cohort who successfully reach full adoption milestones and renew their initial contract term.

**How we measure it:**

$$\text{Completion Rate} = \frac{\text{Customers Who Completed Term}}{\text{Customers Who Activated}} \times 100$$

Key measurement details:
- **Denominator ("activated"):** A customer counts as activated once they finish onboarding *and* perform at least one meaningful action (e.g., running a report or inviting a team member) after the subscription start date. Customers who subscribe but never log in ("no-shows") are exclud

### Finance Team Asks About "CR"

Same question, different org context. "CR" should resolve to **Credit Rating**.

In [9]:
finance_thread = create_thread()
finance_response, finance_msg_id = call_agent(
    "What is CR and how do we measure it?",
    finance_thread,
    system_message="The user is on the Finance team."
)
print("FINANCE TEAM:")
print(finance_response)

FINANCE TEAM:


I'm interpreting **"CR" as Completion Rate**, which is how the Analytics and Customer Success teams use the abbreviation. Note that two other teams use "CR" differently â Finance uses it for **Credit Rating**, and in funnel contexts it can also refer to **Conversion Rate**. If you meant one of those, let me know and I'll switch.

## Completion Rate (CR)

Completion Rate measures the percentage of customers in a cohort who successfully reach full adoption milestones and renew their initial contract term.

**How we measure it:**

$$\text{Completion Rate} = \frac{\text{Customers Who Completed Term}}{\text{Customers Who Activated}} \times 100$$

A few important measurement details:

- **Denominator (activated customers):** A customer counts as "activated" once they complete onboarding and perform at least one meaningful action (e.g., running a report or inviting a team member) after the subscription start date. Customers who subscribe but never log in (no-shows) are exclu

### User Corrects the Disambiguation

Continue the Analytics thread — the user says they actually meant Credit Rating.

In [10]:
# Continue the analytics_thread from above
correction_response, correction_msg_id = call_agent(
    "Actually, I meant Credit Rating. Can you explain that instead?",
    analytics_thread,
    parent_message_id=analytics_msg_id
)
print("CORRECTION:")
print(correction_response)

CORRECTION:


Got it â switching to **Credit Rating**.

## Credit Rating (CR)

**What it is:** Credit Rating is an internal scoring metric used by the Finance team to assess the financial risk profile of a channel partner relationship.

**How it's measured:** It's a composite score on a **1â10 scale**, built from four inputs:
- The partner's **customer growth trajectory** (growing / stable / declining)
- **Contract renewal probability**
- **Revenue concentration risk** â the percentage of total platform revenue coming from this partner
- **Payment history and DSO** (days sales outstanding)

**How Finance uses it:**
- Setting revenue reserves for at-risk partnerships
- Prioritizing contract renegotiation efforts
- Flagging partnerships that may require an impairment assessment

*(Source concept note slug: `credit-rating`)*

One thing to flag: the note defines the four scoring components but doesn't specify how they're weighted or combined into the final 1â10 composite â so if 

---
## Other Examples

### Relationship Traversal

In [11]:
t = create_thread()
response, _ = call_agent("What metrics are related to Cohort and how?", t)
print(response)


I'm interpreting "Cohort" as the glossary concept â a group of customers who begin their subscription during the same time window. The vault states that customer analytics are cohort-based rather than cumulative, so performance metrics are measured at the cohort level and growth targets are set per cohort.

Here are the metrics tied to Cohort and the nature of each relationship:

**Measured at the cohort level (direct):**
- **Completion Rate** â defined explicitly as the percentage of customers *in a cohort* who reach full adoption milestones and renew their initial term. It's only final once the last customer in a cohort has renewed or churned, so the cohort is its unit of measurement.
- **Retention Rate** â the Cohort note names retention (alongside completion) as a performance metric measured per cohort. Retention tracks customers remaining active across milestones within a cohort's subscription period.

**Derived from cohort behavior (indirect):**
- **Lifetime Value (LTV)** 

### Cross-Concept Reasoning

In [12]:
t = create_thread()
response, _ = call_agent("How does plan mix affect our blended metrics?", t)
print(response)

I'll research how plan mix affects blended metrics in your knowledge vault.Let me pull the related concepts around Product Mix to capture any business rules governing blended reporting.The graph traversal tools aren't available, but the concept notes give me a clear picture.

Plan mix â captured in your vault as the **Product Mix** dimension â is the single biggest driver of how your blended (aggregate) metrics behave. Here's how it works and why it matters.

## What "plan mix" means
Product Mix is the distribution of customers (or revenue) across plan types in a reporting period: **Enterprise**, **Professional**, **Starter**, and **Add-On**. A blended metric is any aggregate (activation rate, retention, revenue, LTV, etc.) computed across all of those plans together.

## Why plan mix moves blended metrics
The plan types sit at opposite ends of the trade-off curve, so the *weighting* of each plan changes the blended average even when nothing about any individual plan has changed:



### Entity Search

In [13]:
t = create_thread()
response, _ = call_agent("What data sources feed into our metrics?", t)
print(response)

Based on the knowledge vault, there is currently **one** formally documented data source, and the vault does not catalog others. Here's what feeds into the metrics:

## The documented data source

**Salesforce** is the primary CRM and "source of truth" for the sales funnel, from lead creation through subscription activation. It is replicated to Snowflake via a nightly batch sync (landing by 6:00 AM ET), with near-real-time lead/stage events streamed via Platform Events (~5 min latency). Its key analytics objects land in these raw tables:

| Salesforce Object | Snowflake Table | Primary Use |
|---|---|---|
| Lead | `RAW.SALESFORCE.LEAD` | Funnel top, lead source attribution |
| Opportunity | `RAW.SALESFORCE.OPPORTUNITY` | Funnel progression, conversion analysis |
| Account | `RAW.SALESFORCE.ACCOUNT` | Customer identity, firmographics |
| Subscription__c | `RAW.SALESFORCE.SUBSCRIPTION` | Subscription confirmation, cohort assignment |



## How it feeds the metrics

Salesforce supplies th